In [183]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, sent_tokenize, TweetTokenizer, MWETokenizer
from nltk.probability import FreqDist

from gensim.models import Word2Vec

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "attribute_ruler"])

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt to /Users/nicole/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nicole/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nicole/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Motivation:
Have the meanings of common verbs like "build" or "ship" in viral LinkedIn posts shifted over time?

Within the context of entrepreneurship and corporate social climbing, the language of influencers on platforms like LinkedIn is starkly different from regular, everyday English speech.
- Word embeddings and vectors can track the movement of a word over time or across corpora

### Hypothesis:
Verbs like "build" or "ship" moved from more physical labor-related connotations and contexts and have been diluted with the advent of social network-ifying entrepreneurship and financial success. Adjectives like "great" might also have shifted in meaning from simply a comparative of "good" to mean something closer to "success".

In [111]:
linkedin = pd.read_csv('data/influencers_data.csv')
linkedin_data = linkedin.dropna(subset=['content'])['content']

In [ ]:
# tokenizer = TweetTokenizer()

# initial_tokens = tokenizer.tokenize(linkedin_data[0])

# li_tokens = linkedin_data.map(lambda x: tokenizer.tokenize(x))

# li_tokens = li_tokens.reset_index(drop=True)

# li_tokens = [[i for i in lst if any(c.isalnum() for c in i)] for lst in li_tokens]

# li_tokens = [nltk.pos_tag(lst) for lst in li_tokens]
# li_tokens = [[i for i, tag in lst if tag not in ('NNP', 'NNPS')] for lst in li_tokens]

In [ ]:
texts = linkedin_data.dropna().astype(str).tolist()

li_tokens = []
for doc in nlp.pipe(texts, batch_size=50, n_process=1):
    person_token_idxs = set()
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            for token in ent:
                person_token_idxs.add(token.i)

    tokens = [
        token.text for token in doc
        if token.i not in person_token_idxs
        and token.pos_ != "PROPN"
        and any(c.isalnum() for c in token.text)
    ]
    li_tokens.append(tokens)

In [129]:
def print_word_freq(tokens, target_words):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        # print(len(unpacked_tokens))
        fdist = FreqDist(unpacked_tokens)
    else:
        # print(len(tokens))
        fdist = FreqDist(tokens)
    value_counts = {word : fdist[word] for word in target_words}
    value_counts['total_tokens'] = fdist.N()
    return value_counts

In [201]:
targets = ['build', 'ship', 'great', 'business', 'fellow']
li_target_freq = print_word_freq(li_tokens, targets)
li_target_freq

{'build': 765,
 'ship': 49,
 'great': 1449,
 'business': 2299,
 'fellow': 101,
 'total_tokens': 1633882}

In [105]:
def find_params(tokens):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        tokens_count = len(unpacked_tokens)
    else:
        tokens_count = len(tokens)

    model_params = {
        'tiny' : {
            'vector_size' : 25,
            'min_count' : 3,
            'epochs' : 30,
            'negative' : 15
        },
        'small': {
            'vector_size' : 50,
            'min_count' : 7,
            'epochs' : 20,
            'negative' : 10
        },
        'medium' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 10,
            'negative' : 5
        },
        'large' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 5,
            'negative' : 5
        }
    }

    if tokens_count < 500000:
        model_size = 'tiny'
    elif tokens_count < 5000000:
        model_size = 'small'
    elif tokens_count < 10000000:
        model_size = 'medium'
    else:
        model_size = 'large'

    return model_params[model_size]

In [ ]:
# def create_coha_tokens(decade):
#     dir_path = Path(f'data/coha-samples-text/{decade}')
#     all_tokens = []

#     for f in dir_path.iterdir():
#         if f.is_file():
#             try:
#                 with open(f, 'r', encoding='utf-8') as file:
#                     content = file.readlines()
#                     sentences = sent_tokenize(content[-1].lower().title())
#                     sentences = [word_tokenize(sent) for sent in sentences]
#                     sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
#                     sentences = [nltk.pos_tag(lst) for lst in sentences]
#                     print(sentences)
#                     sentences = [[i[0] for i in lst] for lst in sentences]
#                     sentences = [[i for i, tag in lst if tag not in ('NNP', 'NNPS')] for lst in sentences]
#                     all_tokens.extend(sentences)
#             except Exception as e:
#                 print(f'Could not read {f.name}: {e}')
    
#     return all_tokens

In [230]:
def chunk_text(text, max_chars=100_000):
    chunks = []
    while len(text) > max_chars:
        split_at = text.rfind(' ', 0, max_chars)
        if split_at == -1:
            split_at = max_chars
        chunks.append(text[:split_at])
        text = text[split_at:]
    if text:
        chunks.append(text)
    return chunks

In [217]:
def create_coha_tokens(decade, nlp):
    dir_path = Path(f'data/coha-samples-text/{decade}')
    all_tokens = []

    files = [f for f in dir_path.iterdir() if f.is_file()]
    texts = []
    valid_files = []

    for f in files:
        try:
            with open(f, 'r', encoding='utf-8') as file:
                content = file.readlines()
                text = content[-1]
                if len(text) > 100_000:
                    texts.extend(chunk_text(text))
                else:
                    texts.append(text)
                valid_files.append(f)
        except Exception as e:
            print(f'Could not read {f.name}: {e}')

    for doc in nlp.pipe(texts, batch_size=50):
        for sent in doc.sents:
            tokens = [
                token.text for token in sent
                if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
            ]
            all_tokens.append(tokens)

    return all_tokens

In [172]:
create_coha_tokens('dummy', nlp)

[['The',
  'author',
  'is',
  'indebted',
  'to',
  'one',
  'of',
  'the',
  'novels',
  'of',
  'Le',
  'Brun',
  'for',
  'the',
  'ground',
  'work',
  'of',
  'this',
  'little',
  'comedy'],
 ['DRAMATIS', 'PERSON'],
 ['Philadelphia',
  'Count',
  'Almeyda',
  'Mr.',
  'Robertson',
  'Count',
  'Arandez',
  'Warren',
  'Carlos',
  'Wood',
  'Pacomo',
  'Jefferson',
  'Gusman',
  'Abercrombie',
  'Pedrillo',
  'Durang',
  'Herald',
  'Jackson',
  'Eugenia',
  'Mrs.',
  'Entwistle',
  'Beatrice',
  'Francis',
  'Flora',
  'Claude',
  'Ladies',
  'knights',
  'men',
  'at',
  'arms',
  'pages',
  'servants'],
 ['SCENE',
  'at',
  'the',
  'castle',
  'of',
  'count',
  'Almeyda',
  'in',
  'Catalonia',
  'during',
  'the',
  'thirteenth',
  'century'],
 ['Time', 'twenty', 'four', 'hours'],
 ['Main', 'text', 'ACT', 'I.', 'SCENE', 'I', 'a', 'thick', 'wood'],
 ['Carlos'],
 ['Carlos', 'entering', 'Come', 'along', 'Pacomo'],
 ['Pacomo'],
 ['Pacomo', 'without'],
 ['Here',
  'I',
  'come',

In [ ]:
# def create_coca_tokens(fp):
#     dir_path = Path(fp)
#     all_tokens = []

#     for f in dir_path.iterdir():
#         if f.is_file():
#             try:
#                 with open(f, 'r', encoding='utf-8') as file:
#                     content = file.readlines()
#                     sentences = [re.sub(r'@@\d+', '', s) for s in content]
#                     sentences = [sent_tokenize(s) for s in sentences]
#                     sentences = [word_tokenize(s) for sent in sentences for s in sent]
#                     sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
#                     all_tokens.extend(sentences)
#             except Exception as e:
#                 print(f'Could not read {f.name}: {e}')
    
#     return all_tokens

In [ ]:
def create_coca_tokens(fp, nlp):
    dir_path = Path(fp)
    all_tokens = []

    texts = []
    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    cleaned = [re.sub(r'@@\d+', '', line) for line in content]
                    texts.extend(cleaned)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')

    for doc in nlp.pipe(texts, batch_size=50, n_process=1):
        for sent in doc.sents:
            tokens = [
                token.text for token in sent
                if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
            ]
            all_tokens.append(tokens)

    return all_tokens

In [69]:
def create_model(tokens, params):
    model = Word2Vec(
        sentences=tokens, 
        vector_size=params['vector_size'],
        window=5,            
        min_count=params['min_count'],     
        workers=4,
        sg=1,
        epochs=params['epochs'],
        negative=params['negative']
    )
    return model

In [186]:
li_params = find_params(li_tokens)
li_params

{'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}

In [187]:
li_model = create_model(li_tokens, li_params)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [101]:
def find_neighbors(model, targets):
    # structure:
    # {target_word : [(neighbor1, cos_similarity), (neighbor2, cos_similarity)]}
    neighbors = {
        i : model.wv.most_similar(i) for i in targets
    }
    for n in neighbors:
        print(f"neighbors of '{n}': ")
        print([i[0] for i in neighbors[n]])
    return neighbors

In [188]:
li_neighborhoods = find_neighbors(li_model, targets)

neighbors of 'build': 
['create', 'visualize', 'adopt', 'develop', 'grow', 'drive', 'sustain', 'brand', 'building', 'obtain']
neighbors of 'ship': 
['cruise', 'invent', 'vibe', 'plane', 'pilots', 'Launching', 'unicorn', 'spaceship', 'Galactic', 'vehicle']
neighbors of 'great': 
['fantastic', 'wonderful', 'good', 'amazing', 'excellent', 'awesome', 'superb', 'fabulous', 'incredible', 'nice']
neighbors of 'business': 
['success', 'professionaldevelopment', 'strategy', 'seizetheday', 'businessstrategy', 'https://lnkd.in/gwPNZVi', 'personalbrand', 'brandmarketing', 'solopreneur', 'innovation']
neighbors of 'fellow': 
['contributors', 'honour', 'young', 'Executive', 'Young', 'Meet', 'friends', 'pictured', 'Calling', 'hosts']
neighbors of 'friend': 
['colleague', 'grandmother', 'dad', 'husband', 'dude', 'wife', 'boss', 'mom', 'buddy', 'sister']


In [203]:
def coha_word_neighborhoods(decades, targets):
    tokens = []
    print('Tokenizing data...')
    for i in decades:
        tokens.extend(create_coha_tokens(i, nlp))
    freq_dist = print_word_freq(tokens, targets)
    num_tokens = freq_dist['total_tokens']
    print(f'{num_tokens} tokens created.')
    freq_dist = {t : freq_dist[t] for t in targets}
    print(f'The frequency distribution of target words is: {freq_dist}')
    print('Finding ideal parameters...')
    params = find_params(tokens)
    print(f'Parameters used: {params}')
    print('Creating model...')
    coha_model = create_model(tokens, params)
    print('Words embeddings created!')
    return find_neighbors(coha_model, targets)

In [190]:
pre1850_tokens = create_coha_tokens('pre-1850', nlp)
print_word_freq(pre1850_tokens, targets)

{'build': 5,
 'ship': 110,
 'great': 559,
 'business': 117,
 'fellow': 114,
 'friend': 181,
 'total_tokens': 488917}

In [ ]:
# def create_enron_tokens(fp):
#     dir_path = Path(fp)
#     all_tokens = []

#     for f in dir_path.iterdir():
#         if f.is_file():
#             try:
#                 with open(f, 'r', encoding='utf-8') as file:
#                     content = file.readlines()
#                     sentences = ''.join(content).replace('\n', ' ')
#                     sentences = sent_tokenize(sentences)
#                     sentences = [word_tokenize(s) for s in sentences]
#                     sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
#                     all_tokens.extend(sentences)
#             except Exception as e:
#                 print(f'Could not read {f.name}: {e}')
    
#     return all_tokens

In [220]:
def create_enron_tokens(fp, nlp):
    dir_path = Path(fp)
    all_tokens = []

    texts = []
    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    text = ''.join(content).replace('\n', ' ')
                    if len(text) > 100_000:
                        texts.extend(chunk_text(text))
                    else:
                        texts.append(text)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')

    for doc in nlp.pipe(texts, batch_size=50, n_process=1):
        for sent in doc.sents:
            tokens = [
                token.text for token in sent
                if token.pos_ != "PROPN" and any(c.isalnum() for c in token.text)
            ]
            all_tokens.append(tokens)

    return all_tokens

In [193]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "attribute_ruler"])

In [194]:
coca_tokens = create_coca_tokens('data/coca-samples-text/', nlp)

In [ ]:
coca_params = find_params(coca_tokens)
coca_model = create_model(coca_tokens, coca_params)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

neighbors of 'build': 
['create', 'develop', 'rebuild', 'innovate', 'building', 'maintain', 'enlarge', 'forge', 'infuse', 'transform']
neighbors of 'ship': 
['boat', 'cargo', 'ships', 'Lusitania', 'plane', 'helicopter', 'shuttle', 'planes', 'torpedo', 'astronauts']
neighbors of 'great': 
['good', 'wonderful', 'fantastic', 'terrific', 'terrible', 'perfect', 'amazing', 'tremendous', 'marvelous', 'big']
neighbors of 'business': 
['banking', 'tenants', 'marketing', 'cashing', 'company', '401(k', 'finance', 'businesses', 'ventures', 'venture']
neighbors of 'fellow': 
['sneer', 'former', 'who', 'colleague', 'pastors', 'fellows', 'senior', 'countrymen', 'classmate', 'environmentalist']
neighbors of 'friend': 
['niece', 'friends', 'aunt', 'husband', 'classmate', 'daughter', 'boyfriend', 'mom', 'wife', 'dad']


In [196]:
coca_neighborhood = find_neighbors(coca_model, targets)

neighbors of 'build': 
['create', 'develop', 'rebuild', 'innovate', 'building', 'maintain', 'enlarge', 'forge', 'infuse', 'transform']
neighbors of 'ship': 
['boat', 'cargo', 'ships', 'Lusitania', 'plane', 'helicopter', 'shuttle', 'planes', 'torpedo', 'astronauts']
neighbors of 'great': 
['good', 'wonderful', 'fantastic', 'terrific', 'terrible', 'perfect', 'amazing', 'tremendous', 'marvelous', 'big']
neighbors of 'business': 
['banking', 'tenants', 'marketing', 'cashing', 'company', '401(k', 'finance', 'businesses', 'ventures', 'venture']
neighbors of 'fellow': 
['sneer', 'former', 'who', 'colleague', 'pastors', 'fellows', 'senior', 'countrymen', 'classmate', 'environmentalist']
neighbors of 'friend': 
['niece', 'friends', 'aunt', 'husband', 'classmate', 'daughter', 'boyfriend', 'mom', 'wife', 'dad']


In [233]:
pre1850_neighborhood = coha_word_neighborhoods(['pre-1850'], targets)

Tokenizing data...
488917 tokens created.
The frequency distribution of target words is: {'build': 5, 'ship': 110, 'great': 559, 'business': 117, 'fellow': 114}
Finding ideal parameters...
Parameters used: {'vector_size': 25, 'min_count': 3, 'epochs': 30, 'negative': 15}
Creating model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Words embeddings created!
neighbors of 'build': 
['enmity', 'encourage', 'sink', 'despondency', 'strike', 'retain', 'sit', 'Lyceums', 'endeavouring', 'sin']
neighbors of 'ship': 
['vessel', 'sails', 'Lawrence', 'astern', 'boats', 'enemy', 'leeward', 'Caledonia', 'board', 'sail']
neighbors of 'great': 
['greatest', 'Next', 'sketched', 'compassion', 'declamation', 'pervaded', 'bordering', 'nautical', 'which', 'important']
neighbors of 'business': 
['doing', 'Atlas', 'work', 'problem', 'Jack', 'busily', 'decisively', 'novelty', 'way', 'benefit']
neighbors of 'fellow': 
['natured', 'fellows', 'souled', 'shame', 'man', 'biggest', 'handsomest', 'walloping', 'nothin', 'chaps']


In [204]:
pre1900_neighborhood = coha_word_neighborhoods(['1850s-90s'], targets)

Tokenizing data...
928680 tokens created.
The frequency distribution of target words is: {'build': 29, 'ship': 49, 'great': 1052, 'business': 229, 'fellow': 133}
Finding ideal parameters...
Parameters used: {'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}
Creating model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Words embeddings created!
neighbors of 'build': 
['hire', 'settlements', 'belong', 'farms', 'rust', 'palaces', 'collect', 'selecting', 'clustered', 'trades']
neighbors of 'ship': 
['steamer', 'boat', 'Golden', 'east', 'ladder', 'trip', 'grove', 'issuing', 'crossing', 'cloud']
neighbors of 'great': 
['large', 'vast', 'deal', 'little', 'considerable', 'greatest', 'desperado', 'small', 'comparative', 'essentially']
neighbors of 'business': 
['wealthy', 'locality', 'farming', 'conduct', 'company', 'congregation', 'hotels', 'affairs', 'studies', 'zealous']
neighbors of 'fellow': 
['creature', 'boy', 'minded', 'lion', 'woman', 'conceited', 'poor', 'poke', 'girl', 'nonsense']


In [231]:
d1900_neighborhood = coha_word_neighborhoods(['1900s', '1910s', '1920s', '1930s', '1940s'], targets)

Tokenizing data...
1178372 tokens created.
The frequency distribution of target words is: {'build': 63, 'ship': 296, 'great': 898, 'business': 498, 'fellow': 182}
Finding ideal parameters...
Parameters used: {'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}
Creating model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Words embeddings created!
neighbors of 'build': 
['franchise', 'develop', 'get', 'drive', 'take', 'make', 'secure', 'carry', 'keep', 'construct']
neighbors of 'ship': 
['crew', 'Delilah', 'port', 'aboard', 'boat', 'vessel', 'Peary', 'rig', 'submarine', 'torpoon']
neighbors of 'great': 
['big', 'large', 'considerable', 'little', 'vast', 'sudden', 'dreaded', 'devout', 'wider', 'Danny']
neighbors of 'business': 
['work', 'invested', 'stimulate', 'practice', 'requirements', 'improvement', 'steerage', 'unified', 'market', 'property']
neighbors of 'fellow': 
['boy', 'man', 'gentleman', 'kid', 'chaps', 'lady', 'inspector', 'guy', 'chap', 'beggar']


In [222]:
d1930_neighborhood = coha_word_neighborhoods(['1930s', '1940s'], targets)

Tokenizing data...
521256 tokens created.
The frequency distribution of target words is: {'build': 23, 'ship': 175, 'great': 297, 'business': 166, 'fellow': 39}
Finding ideal parameters...
Parameters used: {'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}
Creating model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Words embeddings created!
neighbors of 'build': 
['carry', 'buy', 'try', 'fix', 'surplus', 'generation', 'sell', 'liquor', 'risk', 'substitute']
neighbors of 'ship': 
['crew', 'dock', 'whaleboat', 'submarine', 'sealmen', "O'Connel", 'Peary', 'battered', 'Base', 'Sandakan']
neighbors of 'great': 
['violent', 'immense', 'good', 'flight', 'bold', 'occasionally', 'talent', 'liquid', 'futile', 'leaves']
neighbors of 'business': 
['objective', 'share', 'Harlem', 'Everyone', 'interests', 'folly', 'cooperation', 'Brazilian', 'fur', 'economy']
neighbors of 'fellow': 
['Sally', 'quarrel', 'blonde', 'sweet', 'young', 'guy', 'Poor', 'stranger', 'fool', 'Midge']


In [232]:
d1950_neighborhood = coha_word_neighborhoods(['1950s', '1960s', '1970s', '1980s', '1990s'], targets)

Tokenizing data...
Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte
Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte
1138864 tokens created.
The frequency distribution of target words is: {'build': 97, 'ship': 83, 'great': 354, 'business': 302, 'fellow': 70}
Finding ideal parameters...
Parameters used: {'vector_size': 50, 'min_count': 7, 'epochs': 20, 'negative': 10}
Creating model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Words embeddings created!
neighbors of 'build': 
['sell', 'computers', 'skilled', 'furnish', 'encourage', 'Luxembourg', 'lend', 'convert', 'await', 'discourage']
neighbors of 'ship': 
['tarantula', 'dismantled', 'relay', 'Tuesday', 'Cacata', 'qualitatively', 'wood', 'ladder', 'fools', 'planks']
neighbors of 'great': 
['pleasant', 'full', 'strong', 'deal', 'spy', 'Girl', 'ironic', 'coincidence', 'Victorian', 'spreading']
neighbors of 'business': 
['money', 'cheap', 'city', 'investment', 'counselors', 'budgets', 'invest', 'wealth', 'grades', 'investments']
neighbors of 'fellow': 
['Rick', 'Nyama', 'friend', 'Dutchman', 'curator', 'reporter', 'Judah', 'politician', 'lad', 'Wright']


In [226]:
enron_tokens = create_enron_tokens('data/enronsent', nlp)

Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


In [227]:
enron_params = find_params(enron_tokens)
enron_model = create_model(enron_tokens, enron_params)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [229]:
enron_neighborhood = find_neighbors(enron_model, targets)

neighbors of 'build': 
['develop', 'operate', 'modular', 'halt', 'cooled', 'hinder', 'construct', 'rebuild', 'dynamically', 'cogeneration']
neighbors of 'ship': 
['dawn', 'Suburban', 'unloading', 'inspect', 'widen', 'bend', 'occupy', 'bikes', 'fluids', 'roads']
neighbors of 'great': 
['good', 'wonderful', 'fantastic', 'nice', 'terrific', 'perfect', 'fun', 'neat', 'fabulous', 'cool']
neighbors of 'business': 
['business=', 'operations', 'successfu=', 'assum=', 'activities', 'rapport', 'initiatives', 'business=20', 'businesses', 'marketing']
neighbors of 'fellow': 
['veterans', 'influential', 'classmates', 'heroes', 'longtime', 'supporters', 'overheard', 'researcher', 'inexperienced', 'congressman']


## Results:
#### LinkedIn word neighborhoods:
- neighbors of 'build': 
  - ['create', 'visualize', 'adopt', 'develop', 'grow', 'drive', 'sustain', 'brand', 'building', 'obtain']
- neighbors of 'ship': 
  - ['cruise', 'invent', 'vibe', 'plane', 'pilots', 'Launching', 'unicorn', 'spaceship', 'Galactic', 'vehicle']
- neighbors of 'great': 
  - ['fantastic', 'wonderful', 'good', 'amazing', 'excellent', 'awesome', 'superb', 'fabulous', 'incredible', 'nice']
- neighbors of 'business': 
  - ['success', 'professionaldevelopment', 'strategy', 'seizetheday', 'businessstrategy', 'https://lnkd.in/gwPNZVi', 'personalbrand', 'brandmarketing', 'solopreneur', 'innovation']
- neighbors of 'fellow': 
  - ['contributors', 'honour', 'young', 'Executive', 'Young', 'Meet', 'friends', 'pictured', 'Calling', 'hosts']

#### Pre-1850s word neighborhoods:
- neighbors of 'build': 
  - ['enmity', 'encourage', 'sink', 'despondency', 'strike', 'retain', 'sit', 'Lyceums', 'endeavouring', 'sin']
- neighbors of 'ship': 
  - ['vessel', 'sails', 'Lawrence', 'astern', 'boats', 'enemy', 'leeward', 'Caledonia', 'board', 'sail']
- neighbors of 'great': 
  - ['greatest', 'Next', 'sketched', 'compassion', 'declamation', 'pervaded', 'bordering', 'nautical', 'which', 'important']
- neighbors of 'business': 
  - ['doing', 'Atlas', 'work', 'problem', 'Jack', 'busily', 'decisively', 'novelty', 'way', 'benefit']
- neighbors of 'fellow': 
  - ['natured', 'fellows', 'souled', 'shame', 'man', 'biggest', 'handsomest', 'walloping', 'nothin', 'chaps']

#### 1850-1900s word neighborhoods:
- neighbors of 'build': 
  - ['hire', 'settlements', 'belong', 'farms', 'rust', 'palaces', 'collect', 'selecting', 'clustered', 'trades']
- neighbors of 'ship': 
  - ['steamer', 'boat', 'Golden', 'east', 'ladder', 'trip', 'grove', 'issuing', 'crossing', 'cloud']
- neighbors of 'great': 
  - ['large', 'vast', 'deal', 'little', 'considerable', 'greatest', 'desperado', 'small', 'comparative', 'essentially']
- neighbors of 'business': 
  - ['wealthy', 'locality', 'farming', 'conduct', 'company', 'congregation', 'hotels', 'affairs', 'studies', 'zealous']
- neighbors of 'fellow': 
  - ['creature', 'boy', 'minded', 'lion', 'woman', 'conceited', 'poor', 'poke', 'girl', 'nonsense']

#### 1900-1950s word neighborhoods:
- neighbors of 'build': 
  - ['franchise', 'develop', 'get', 'drive', 'take', 'make', 'secure', 'carry', 'keep', 'construct']
- neighbors of 'ship': 
  - ['crew', 'Delilah', 'port', 'aboard', 'boat', 'vessel', 'Peary', 'rig', 'submarine', 'torpoon']
- neighbors of 'great': 
  - ['big', 'large', 'considerable', 'little', 'vast', 'sudden', 'dreaded', 'devout', 'wider', 'Danny']
- neighbors of 'business': 
  - ['work', 'invested', 'stimulate', 'practice', 'requirements', 'improvement', 'steerage', 'unified', 'market', 'property']
- neighbors of 'fellow': 
  - ['boy', 'man', 'gentleman', 'kid', 'chaps', 'lady', 'inspector', 'guy', 'chap', 'beggar']

#### 1950-2000s word neighborhoods:
- neighbors of 'build': 
  - ['sell', 'computers', 'skilled', 'furnish', 'encourage', 'Luxembourg', 'lend', 'convert', 'await', 'discourage']
- neighbors of 'ship': 
  - ['tarantula', 'dismantled', 'relay', 'Tuesday', 'Cacata', 'qualitatively', 'wood', 'ladder', 'fools', 'planks']
- neighbors of 'great': 
  - ['pleasant', 'full', 'strong', 'deal', 'spy', 'Girl', 'ironic', 'coincidence', 'Victorian', 'spreading']
- neighbors of 'business': 
  - ['money', 'cheap', 'city', 'investment', 'counselors', 'budgets', 'invest', 'wealth', 'grades', 'investments']
- neighbors of 'fellow': 
  - ['Rick', 'Nyama', 'friend', 'Dutchman', 'curator', 'reporter', 'Judah', 'politician', 'lad', 'Wright']

#### Contemporary (1990-2019) word neighborhoods:
- neighbors of 'build': 
  - ['create', 'develop', 'rebuild', 'innovate', 'building', 'maintain', 'enlarge', 'forge', 'infuse', 'transform']
- neighbors of 'ship': 
  - ['boat', 'cargo', 'ships', 'Lusitania', 'plane', 'helicopter', 'shuttle', 'planes', 'torpedo', 'astronauts']
- neighbors of 'great': 
  - ['good', 'wonderful', 'fantastic', 'terrific', 'terrible', 'perfect', 'amazing', 'tremendous', 'marvelous', 'big']
- neighbors of 'business': 
  - ['banking', 'tenants', 'marketing', 'cashing', 'company', '401(k', 'finance', 'businesses', 'ventures', 'venture']
- neighbors of 'fellow': 
  - ['sneer', 'former', 'who', 'colleague', 'pastors', 'fellows', 'senior', 'countrymen', 'classmate', 'environmentalist']

#### EnronSent word neighborhoods:
- neighbors of 'build': 
  - ['develop', 'operate', 'modular', 'halt', 'cooled', 'hinder', 'construct', 'rebuild', 'dynamically', 'cogeneration']
- neighbors of 'ship': 
  - ['dawn', 'Suburban', 'unloading', 'inspect', 'widen', 'bend', 'occupy', 'bikes', 'fluids', 'roads']
- neighbors of 'great': 
  - ['good', 'wonderful', 'fantastic', 'nice', 'terrific', 'perfect', 'fun', 'neat', 'fabulous', 'cool']
- neighbors of 'business': 
  - ['business=', 'operations', 'successfu=', 'assum=', 'activities', 'rapport', 'initiatives', 'business=20', 'businesses', 'marketing']
- neighbors of 'fellow': 
  - ['veterans', 'influential', 'classmates', 'heroes', 'longtime', 'supporters', 'overheard', 'researcher', 'inexperienced', 'congressman']

## Hurdles faced:
1. Many proper nouns/names included in embedding step needed to be filtered out
- Initially used `nltk`, but switched to `SpaCy`
- `SpaCy` took much longer to run than `nltk` and needed files with >100k characters to be batch processed
2. Some target words only appeared in each decade's sample texts a handful of times
- Good embeddings seemed to need at least 50-100 minimum occurrences for `Word2Vec` to learn meaningful relationships
- Sample text data from COHA might have been skewed towards certain domains more than others as they were randomly chosen from BYU's COHA corpus
3. Data sparsity -- each decade's worth of text data from the COHA sample corpus contained around 120k-300k tokens
- Compared to the millions or billions needed for modern embedding models, meaningful relationships would, again, be more difficult to learn
- Solution: grouped five decades together -- potentially lower resolution analysis of word neighborhoods between decades, but models were better fitted and embeddings less random

## Future Directions:
1. Create word cloud data visualizations to better show timeline of target words and their nearest neighbors
2. Lemmatize tokens to find if embeddings improve
- Currently, words like `'build'` and `'building'` are treated as two separate tokens despite both being verbs of the same **lemma**
3. Compare LinkedIn embeddings with EnronSent's embeddings
- Both are corporate-speak spaces -- how similar would these target words' neighbors be despite a two decade gap?
4. Find more data to more robustly examine whether target words have shifted in meaning over time